# CHAPTER 8 : Data Wrangling: Join, Combine, and Reshape

-----

In many applications, data may be spread across a number of files or databases or be
arranged in a form that is not easy to analyze. This chapter focuses on tools to help
combine, join, and rearrange data.

First, I introduce the concept of hierarchical indexing in pandas, which is used extensively
in some of these operations. I then dig into the particular data manipulations.
You can see various applied usages of these tools in Chapter 14.

## Basic Imports and Set ups

In [4]:
import pandas as pd
import numpy as np
import os
from faker import Faker

This is a function to render dataframes as tables in the Jupyter Notebook.

In [5]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)



## 8.1 Hierarchical Indexing

*Hierarchical indexing* is an important feature of pandas that enables you to have multiple
(two or more) index levels on an axis. Somewhat abstractly, it provides a way for
you to work with higher dimensional data in a lower dimensional form. Let’s start
with a simple example; create a Series with a list of lists (or arrays) as the index:

In [6]:
data = pd.Series(np.random.randn(9),index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],[1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [7]:
data # You can see 2 indexes for Series.

a  1   -0.688790
   2    0.793909
   3    0.873270
b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
d  2    0.474364
   3   -0.680989
dtype: float64

What you’re seeing is a prettified view of a Series with a MultiIndex as its index. The
“gaps” in the index display mean “use the label directly above”:

In [8]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [13]:
data.index.levels

FrozenList([['a', 'b', 'c', 'd'], [1, 2, 3]])

In [12]:
data.index.codes  # Codes was earlier called Labels.

FrozenList([[0, 0, 0, 1, 1, 2, 2, 3, 3], [0, 1, 2, 0, 2, 0, 1, 1, 2]])

With a hierarchically indexed object, so-called partial indexing is possible, enabling
you to concisely select subsets of the data

In [9]:
data['b']

1    1.889498
3   -0.951723
dtype: float64

---------

##### ✅ What does **`labels`** mean in this `MultiIndex`?

When you create a **`MultiIndex`**, pandas internally stores it in a **factorized (encoded) form** to save memory and improve performance.

It does **NOT store the actual index values repeatedly.**
Instead, it stores:

* **`levels` → the unique values at each index level**
* **`labels` → integer pointers (positions) telling which level value to use for each row**

---

##### 🔵 In your example

You created:

```python
data = pd.Series(
    np.random.randn(9),
    index=[
        ['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
        [1, 2, 3, 1, 3, 1, 2, 2, 3]
    ]
)
```

And pandas shows:

```text
MultiIndex(
  levels=[['a','b','c','d'], [1,2,3]],
  labels=[[0,0,0,1,1,2,2,3,3], [0,1,2,0,2,0,1,1,2]]
)
```

---

##### 🟢 Step-1: Understand `levels`

These are the **unique sorted values** for each index level.

Level-0:

```
['a','b','c','d']
```

Level-1:

```
[1,2,3]
```

---

##### 🟢 Step-2: Understand `labels`

`labels` are **integer arrays that reference positions inside the `levels` list.**

Think of them as:

> “Which value from `levels` should be used at this row?”

---

###### ⭐ Level-0 labels

```
[0,0,0,1,1,2,2,3,3]
```

Mapping using:

```
levels[0] = ['a','b','c','d']
```

So:

| Label | Means |
| ----- | ----- |
| 0 →   | 'a'   |
| 1 →   | 'b'   |
| 2 →   | 'c'   |
| 3 →   | 'd'   |

Thus pandas reconstructs:

```
a a a b b c c d d
```

---

###### ⭐ Level-1 labels

```
[0,1,2,0,2,0,1,1,2]
```

Mapping using:

```
levels[1] = [1,2,3]
```

So:

| Label | Means |
| ----- | ----- |
| 0 →   | 1     |
| 1 →   | 2     |
| 2 →   | 3     |

Thus pandas reconstructs:

```
1 2 3 1 3 1 2 2 3
```

---

##### 🎯 Final Reconstruction Logic

For row `i`:

```
actual_index_value =
(
   levels[0][ labels[0][i] ],
   levels[1][ labels[1][i] ]
)
```

Example → row 4

```
labels → (1 , 2)

levels[0][1] → 'b'
levels[1][2] → 3

→ index = ('b', 3)
```

---

##### 🚀 Why pandas does this (VERY important Data Science insight)

This is basically **categorical / dictionary encoding of index values.**

Benefits:

✅ avoids storing repeated strings
✅ faster grouping / sorting
✅ lower memory
✅ vectorized operations on integer codes

This exact idea appears in:

* pandas `Categorical`
* database dictionary encoding
* parquet column encoding
* feature engineering pipelines

---

##### ⚠️ Modern pandas note (important)

In **new pandas versions**, `labels` is renamed to:

```
codes
```

So you may see:

```python
data.index.codes
```

instead of:

```
data.index.labels
```

Same concept. Only name changed.

---



With a hierarchically indexed object, so-called partial indexing is possible, enabling
you to concisely select subsets of the data:

In [14]:
data['b']

1    1.889498
3   -0.951723
dtype: float64

In [15]:
data['b':'c']

b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
dtype: float64

In [16]:
data.loc[['b', 'd']]

b  1    1.889498
   3   -0.951723
d  2    0.474364
   3   -0.680989
dtype: float64

Selection is even possible from an “inner” level:

In [18]:
data.loc[:, 2]

a    0.793909
c   -0.152598
d    0.474364
dtype: float64

Hierarchical indexing plays an important role in reshaping data and group-based
operations like forming a pivot table. For example, you could rearrange the data into
a DataFrame using its `unstack` method:

In [19]:
data.unstack()

,1,2,3
a,-0.688790,0.793909,0.873270
b,1.889498,NaN,-0.951723
c,-0.149447,-0.152598,NaN
d,NaN,0.474364,-0.680989


The inverse operation of `unstack` is `stack`:

In [21]:
data.unstack().stack()

a  1   -0.688790
   2    0.793909
   3    0.873270
b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
d  2    0.474364
   3   -0.680989
dtype: float64

`stack` and `unstack`will be explored in more detail later in this chapter.

----

In [55]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],columns=[['Ohio', 'Ohio', 'Colorado'],['Green', 'Red', 'Green']])

With a DataFrame, either axis can have a hierarchical index:

In [56]:
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

The hierarchical levels can have names (as strings or any Python objects). If so, these
will show up in the console output:

In [57]:
frame.index.names = ['key1', 'key2']

In [58]:
frame.columns.names = ['state', 'color']

In [59]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

> Be careful to distinguish the index names 'state' and 'color'
from the row labels.

With partial column indexing you can similarly select groups of columns:

In [60]:
frame['Ohio']

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

A MultiIndex can be created by itself and then reused; the columns in the preceding
DataFrame with level names could be created like this:

In [63]:
mindex_for_cols = pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
names=['state', 'color'])

In [64]:
mindex_for_rows = pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
names=['key1', 'key2'])

In [68]:
frame2 = pd.DataFrame(np.arange(12).reshape((4, 3)),index= mindex_for_rows,columns=mindex_for_cols)

In [70]:
frame2

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

### Reordering and Sorting Levels

At times you will need to rearrange the order of the levels on an axis or sort the data
by the values in one specific level. The `swaplevel` takes two level numbers or names
and returns a new object with the levels interchanged (but the data is otherwise
unaltered):

In [71]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [72]:
frame.swaplevel('key1', 'key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

`sort_index`, on the other hand, sorts the data using only the values in a single level.
When swapping levels, it’s not uncommon to also use `sort_index` so that the result is
lexicographically sorted by the indicated level:

In [74]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [76]:
frame.sort_index(level=0,ascending= False)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
b    2        9  10       11
     1        6   7        8
a    2        3   4        5
     1        0   1        2

In [77]:
frame.sort_index(level=1, ascending= False)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
b    2        9  10       11
a    2        3   4        5
b    1        6   7        8
a    1        0   1        2

In [78]:
frame.swaplevel(0, 1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

> Data selection performance is much better on hierarchically
indexed objects if the index is lexicographically sorted starting with
the outermost level—that is, the result of calling
`sort_index(level=0)` or `sort_index()`.

### Summary Statistics by Level

Many descriptive and summary statistics on DataFrame and Series have a level
option in which you can specify the level you want to aggregate by on a particular
axis. Consider the above DataFrame; we can aggregate by level on either the rows or
columns like so:

In [79]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [80]:
frame.sum(level='key2')

TypeError: sum() got an unexpected keyword argument 'level'

This is failing because in newer version as of the time of this notebook, this option is deprecated. 

##### ✅ Why you are getting this error

You are seeing:

```
TypeError: sum() got an unexpected keyword argument 'level'
```

because **in modern pandas versions (`>= 1.3` and fully removed later)** the `level=` argument in reduction functions like:

```
frame.sum(level='key2')
```

has been **deprecated and removed.**

Earlier (very old pandas — like in the book/course you are studying), this worked because pandas allowed:

```
DataFrame.sum(level=...)
```

to perform aggregation across a **MultiIndex level.**

That API is now replaced with **`groupby(level=...)`.**

---

##### ✅ Correct Modern Pandas Way (Very Important)

You must now write:

```
frame2.groupby(level='key2').sum()
```

This is the **exact modern equivalent** of:

```
frame2.sum(level='key2')
```

---

##### ✅ Let’s Understand What is Happening Conceptually

Your dataframe:

* Rows → MultiIndex (`key1`, `key2`)
* Columns → MultiIndex (`state`, `color`)

When you do:

```
groupby(level='key2')
```

You are saying:

👉 **“Group rows that have the same `key2` value and then sum them.”**

So rows:

```
(a,1)
(b,1)
```

will be grouped together

and rows:

```
(a,2)
(b,2)
```

will be grouped together

Then sum is computed column-wise.

---

##### ✅ Full Working Example

```
frame2.groupby(level='key2').sum()
```

Result structure:

* Index → only `key2`
* Columns → unchanged MultiIndex columns

---

##### ✅ If You Wanted Column Level Aggregation

Very powerful point (senior-level understanding 🙂)

If you wanted to aggregate **across column MultiIndex level**, you would do:

```
frame2.groupby(level='state', axis=1).sum()
```

So now pandas groups **columns instead of rows.**

---

##### ✅ Why Pandas Removed `level=` from `.sum()`

Architectural reason:

* Old API → hidden grouping behaviour inside reduction
* New API → **explicit grouping step**
* Cleaner mental model
* Consistent with:

```
groupby → aggregate
```

This aligns pandas more with SQL / distributed systems / dataframe engines.

---

##### ⭐ Senior Insight (Very Useful for You in Data Science)

Whenever you see:

```
something.sum(level=...)
```

Immediately translate in your head to:

```
something.groupby(level=...).sum()
```

This will save you **huge debugging time** when reading older books / notebooks / courses.

---


In [81]:
frame2.groupby(level='key2').sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

Under the hood, this utilizes pandas’s groupby machinery, which will be discussed in
more detail later in the book.

### Indexing with a DataFrame’s columns

It’s not unusual to want to use one or more columns from a DataFrame as the row
index; alternatively, you may wish to move the row index into the DataFrame’s columns.
Here’s an example DataFrame:

In [84]:
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two',
                            'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})

In [85]:
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame’s `set_index` function will create a new DataFrame using one or more of
its columns as the index:

In [86]:
frame2 = frame.set_index(['c', 'd'])

In [87]:
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

By default the columns are removed from the DataFrame, though you can leave them
in:

In [88]:
frame.set_index(['c', 'd'], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

`reset_index`, on the other hand, does the opposite of `set_index`; the hierarchical
index levels are moved into the columns:



---

## 8.2 Combining and Merging Datasets

Data contained in pandas objects can be combined together in a number of ways:

• `pandas.merge` connects rows in DataFrames based on one or more keys. This
will be familiar to users of SQL or other relational databases, as it implements
database join operations.

• `pandas.concat` concatenates or “stacks” together objects along an axis.

• The `combine_first` instance method enables splicing together overlapping data
to fill in missing values in one object with values from another.

I will address each of these and give a number of examples. They’ll be utilized in
examples throughout the rest of the book.

### Database-Style DataFrame Joins

*Merge* or *join* operations combine datasets by linking rows using one or more keys.
These operations are central to relational databases (e.g., SQL-based). The `merge`
function in pandas is the main entry point for using these algorithms on your data.

Let’s start with a simple example:

In [100]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],'data1': range(7)})

In [101]:
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],'data2': range(3)})

In [102]:
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [103]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


This is an example of a many-to-one join; the data in `df1` has multiple rows labeled a
and b, whereas `df2` has only one row for each value in the key column. Calling `merge`
with these objects we obtain:

In [105]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


Note that I didn’t specify which column to join on. If that information is not specified,
**merge uses the overlapping column names as the keys**. It’s a good practice to
specify explicitly, though:

In [107]:
pd.merge(df1, df2, on='key')

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


If the column names are different in each object, you can specify them separately:

In [108]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],'data1': range(7)})

In [109]:
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'],'data2': range(3)})

In [110]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


You may notice that the `'c'` and `'d'` values and associated data are missing from the
result. By default merge does an `'inner'` join; the keys in the result are the intersection,
or the common set found in both tables. Other possible options are `'left'`,
`'right'`, and `'outer'`. The outer join takes the union of the keys, combining the effect of applying both left and right joins:

In [111]:
pd.merge(df1, df2, how='outer')

,key,data1,data2
0,a,2.0,0.0
1,a,4.0,0.0
2,a,5.0,0.0
3,b,0.0,1.0
4,b,1.0,1.0
5,b,6.0,1.0
6,c,3.0,NaN
7,d,NaN,2.0


See Table 8-1 for a summary of the options for how.

In [113]:
columns = ["Option", "Behavior"]

rows = [
["inner","Use only the key combinations observed in both tables"],
["left","Use all key combinations found in the left table"],
["right","Use all key combinations found in the right table"],
["outer","Use all key combinations observed in both tables together"]
]

render_book_table(
    "Table 8-1. Different Join Types with how Argument",
    columns,
    rows
)

Option,Behavior
inner,Use only the key combinations observed in both tables
left,Use all key combinations found in the left table
right,Use all key combinations found in the right table
outer,Use all key combinations observed in both tables together


Many-to-many merges have well-defined, though not necessarily intuitive, behavior.
Here’s an example:

In [116]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],'data1': range(6)})

In [117]:
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],'data2': range(5)})

In [118]:
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [119]:
df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [120]:
pd.merge(df1, df2, on='key', how='left')

,key,data1,data2
0,b,0,1.0
1,b,0,3.0
2,b,1,1.0
3,b,1,3.0
4,a,2,0.0
5,a,2,2.0
6,c,3,NaN
7,a,4,0.0
8,a,4,2.0
9,b,5,1.0


Many-to-many joins form the Cartesian product of the rows. Since there were three
`'b'` rows in the left DataFrame and two in the right one, there are six `'b'` rows in the
result. The join method only affects the distinct key values appearing in the result:

In [121]:
pd.merge(df1, df2, how='inner')

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


----

To merge with multiple keys, pass a list of column names:

In [123]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],'key2': ['one', 'two', 'one'],'lval': [1, 2, 3]})

In [124]:
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],'key2': ['one', 'one', 'one', 'two'],'rval': [4, 5, 6, 7]})

In [125]:
left

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3


In [126]:
right

,key1,key2,rval
0,foo,one,4
1,foo,one,5
2,bar,one,6
3,bar,two,7


In [127]:
pd.merge(left, right, on=['key1', 'key2'], how='outer')

,key1,key2,lval,rval
0,bar,one,3.0,6.0
1,bar,two,NaN,7.0
2,foo,one,1.0,4.0
3,foo,one,1.0,5.0
4,foo,two,2.0,NaN


To determine which key combinations will appear in the result depending on the
choice of merge method, think of the multiple keys as forming an array of tuples to
be used as a single join key (even though it’s not actually implemented that way).

> When you’re joining columns-on-columns, the indexes on the
passed DataFrame objects are discarded.

A last issue to consider in merge operations is the treatment of overlapping column
names. While you can address the overlap manually (see the earlier section on
renaming axis labels), `merge` has a `suffixes` option for specifying strings to append
to overlapping names in the left and right DataFrame objects:

In [128]:
pd.merge(left, right, on='key1')

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


In [131]:
pd.merge(left, right, on='key2', how='outer')

,key1_x,key2,lval,key1_y,rval
0,foo,one,1,foo,4
1,foo,one,1,foo,5
2,foo,one,1,bar,6
3,bar,one,3,foo,4
4,bar,one,3,foo,5
5,bar,one,3,bar,6
6,foo,two,2,bar,7


In [132]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right'))

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


See Table 8-2 for an argument reference on `merge`. Joining using the DataFrame’s row
index is the subject of the next section.



In [135]:
columns = ["Argument", "Description"]

rows = [
["left","DataFrame to be merged on the left side"],
["right","DataFrame to be merged on the right side"],
["how","One of 'inner', 'outer', 'left', or 'right'; defaults to 'inner'"],
["on","Column names to join on; must exist in both DataFrames"],
["left_on","Columns in left DataFrame to use as join keys"],
["right_on","Analogous to left_on for right DataFrame"],
["left_index","Use row index in left as join key (or keys if MultiIndex)"],
["right_index","Analogous to left_index"],
["sort","Sort merged data lexicographically by join keys; True by default"],
["suffixes","Tuple of suffixes for overlapping column names; default ('_x','_y')"],
["copy","If False, avoid copying data in some exceptional cases"],
["indicator","Adds column '_merge' showing row origin: 'left_only', 'right_only', or 'both'"]
]

render_book_table(
    "Table 8-2. merge Function Arguments",
    columns,
    rows
)

Argument,Description
left,DataFrame to be merged on the left side
right,DataFrame to be merged on the right side
how,"One of 'inner', 'outer', 'left', or 'right'; defaults to 'inner'"
on,Column names to join on; must exist in both DataFrames
left_on,Columns in left DataFrame to use as join keys
right_on,Analogous to left_on for right DataFrame
left_index,Use row index in left as join key (or keys if MultiIndex)
right_index,Analogous to left_index
sort,Sort merged data lexicographically by join keys; True by default
suffixes,"Tuple of suffixes for overlapping column names; default ('_x','_y')"


### Merging on Index

In some cases, the merge key(s) in a DataFrame will be found in its index. In this
case, you can pass `left_index=True` or `right_index=True` (or both) to indicate that
the index should be used as the merge key:

In [138]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],'value': range(6)})

In [139]:
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])

In [140]:
left1

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [141]:
right1

,group_val
a,3.5
b,7.0


In [142]:
pd.merge(left1, right1, left_on='key', right_index=True)

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


Since the default merge method is to intersect the join keys, you can instead form the
union of them with an outer join:

In [143]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


----

With hierarchically indexed data, things are more complicated, as joining on index is
implicitly a multiple-key merge:

In [145]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio','Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': np.arange(5.)})
                      

In [146]:
righth = pd.DataFrame(np.arange(12).reshape((6, 2)),
                      index=[['Nevada', 'Nevada', 'Ohio', 'Ohio','Ohio', 'Ohio'],
                             [2001, 2000, 2000, 2000, 2001, 2002]],
                      columns=['event1', 'event2'])

In [147]:
lefth

,key1,key2,data
0,Ohio,2000,0.0
1,Ohio,2001,1.0
2,Ohio,2002,2.0
3,Nevada,2001,3.0
4,Nevada,2002,4.0


In [149]:
righth

event1  event2
Nevada 2001       0       1
       2000       2       3
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11

In this case, you have to indicate multiple columns to merge on as a list (note the
handling of duplicate index values with `how='outer'`):

In [151]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True)

,key1,key2,data,event1,event2
0,Ohio,2000,0.0,4,5
0,Ohio,2000,0.0,6,7
1,Ohio,2001,1.0,8,9
2,Ohio,2002,2.0,10,11
3,Nevada,2001,3.0,0,1


In [152]:
pd.merge(lefth, righth, left_on=['key1', 'key2'],right_index=True, how='outer')

,key1,key2,data,event1,event2
4,Nevada,2000,NaN,2.0,3.0
3,Nevada,2001,3.0,0.0,1.0
4,Nevada,2002,4.0,NaN,NaN
0,Ohio,2000,0.0,4.0,5.0
0,Ohio,2000,0.0,6.0,7.0
1,Ohio,2001,1.0,8.0,9.0
2,Ohio,2002,2.0,10.0,11.0


Using the indexes of both sides of the merge is also possible:

In [155]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])

In [156]:
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])

In [157]:
left2

,Ohio,Nevada
a,1.0,2.0
c,3.0,4.0
e,5.0,6.0


In [158]:
right2

,Missouri,Alabama
b,7.0,8.0
c,9.0,10.0
d,11.0,12.0
e,13.0,14.0


In [159]:
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


DataFrame has a convenient `join` instance for merging by index. It can also be used
to combine together many DataFrame objects having the same or similar indexes but
non-overlapping columns. In the prior example, we could have written:


In [162]:
left2.join(right2, how='outer')

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


In part for legacy reasons (i.e., much earlier versions of pandas), DataFrame’s `join`
method performs a left join on the join keys, exactly preserving the left frame’s row
index. It also supports joining the index of the passed DataFrame on one of the columns
of the calling DataFrame:

In [163]:
left1

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [164]:
right1

,group_val
a,3.5
b,7.0


In [165]:
left1.join(right1, on='key')

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


Lastly, for simple index-on-index merges, you can pass a list of DataFrames to join as
an alternative to using the more general`concat` function described in the next
section:

In [166]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=['a', 'c', 'e', 'f'],
                       columns=['New York', 'Oregon'])

In [167]:
another

,New York,Oregon
a,7.0,8.0
c,9.0,10.0
e,11.0,12.0
f,16.0,17.0


In [168]:
left2

,Ohio,Nevada
a,1.0,2.0
c,3.0,4.0
e,5.0,6.0


In [169]:
right2

,Missouri,Alabama
b,7.0,8.0
c,9.0,10.0
d,11.0,12.0
e,13.0,14.0


In [170]:
left2.join([right2, another]) # Here only left2 has left join and from rest only common index values taken.

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0


In [171]:
left2.join([right2, another], how='outer')

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0
b,NaN,NaN,7.0,8.0,NaN,NaN
d,NaN,NaN,11.0,12.0,NaN,NaN
f,NaN,NaN,NaN,NaN,16.0,17.0


### Concatenating Along an Axis

Another kind of data combination operation is referred to interchangeably as concatenation,
binding, or stacking. NumPy’s `concatenate` function can do this with
NumPy arrays

In [173]:
arr = np.arange(12).reshape((3, 4))

In [174]:
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [175]:
np.concatenate([arr, arr], axis=1)

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

In the context of pandas objects such as Series and DataFrame, having labeled axes
enable you to further generalize array concatenation. In particular, you have a number
of additional things to think about:

• If the objects are indexed differently on the other axes, should we combine the
distinct elements in these axes or use only the shared values (the intersection)?

• Do the concatenated chunks of data need to be identifiable in the resulting
object?

• Does the “concatenation axis” contain data that needs to be preserved? In many
cases, the default integer labels in a DataFrame are best discarded during
concatenation.

The `concat` function in pandas provides a consistent way to address each of these
concerns. I’ll give a number of examples to illustrate how it works. Suppose we have
three Series with no index overlap:

In [176]:
s1 = pd.Series([0, 1], index=['a', 'b'])

In [177]:
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])